In [24]:
source ../modules/setup.tcl

set year 2021
set day 21

aoc::get-puzzle $year $day 1
# aoc::get-puzzle $year $day 2
set input [string trim [aoc::get-input $year $day]]
jupyter::display "text/markdown" "## Input\n```\n[string range $input 0 20]...\n```";

(cached)

--- Day 21: Dirac Dice --- There's not much to do as you slowly descend to the bottom of the ocean. The submarine computer challenges you to a nice game of Dirac Dice . 
 This game consists of a single die , two pawns , and a game board with a circular track containing ten spaces marked 1 through 10 clockwise. Each player's starting space is chosen randomly (your puzzle input). Player 1 goes first. 
 Players take turns moving. On each player's turn, the player rolls the die three times and adds up the results. Then, the player moves their pawn that many times forward around the track (that is, moving clockwise on spaces in order of increasing value, wrapping back around to 1 after 10 ). So, if a player is on space 7 and they roll 2 , 2 , and 1 , they would move forward 5 times, to spaces 8 , 9 , 10 , 1 , and finally stopping on 2 . 
 After each player moves, they increase their score by the value of the space their pawn stopped on. Players' scores start at 0 . So, if the first player starts on space 7 and rolls a total of 5 , they would stop on space 2 and add 2 to their score (for a total score of 2 ). The game immediately ends as a win for any player whose score reaches at least 1000 
 . 
 Since the first game is a practice game, the submarine opens a compartment labeled deterministic dice and a 100-sided die falls out. This die always rolls 1 first, then 2 , then 3 , and so on up to 100 , after which it starts over at 1 again. Play using this die. 
 For example, given these starting positions: 
 Player 1 starting position: 4
Player 2 starting position: 8
 
 This is how the game would go: 
 
 Player 1 rolls 1 + 2 + 3 and moves to space 10 for a total score of 10 . 
 Player 2 rolls 4 + 5 + 6 and moves to space 3 for a total score of 3 . 
 Player 1 rolls 7 + 8 + 9 and moves to space 4 for a total score of 14 . 
 Player 2 rolls 10 + 11 + 12 and moves to space 6 for a total score of 9 . 
 Player 1 rolls 13 + 14 + 15 and moves to space 6 for a total score of 20 . 
 Player 2 rolls 16 + 17 + 18 and moves to space 7 for a total score of 16 . 
 Player 1 rolls 19 + 20 + 21 and moves to space 6 for a total score of 26 . 
 Player 2 rolls 22 + 23 + 24 and moves to space 6 for a total score of 22 . 
 
 ...after many turns... 
 
 Player 2 rolls 82 + 83 + 84 and moves to space 6 for a total score of 742 . 
 Player 1 rolls 85 + 86 + 87 and moves to space 4 for a total score of 990 . 
 Player 2 rolls 88 + 89 + 90 and moves to space 3 for a total score of 745 . 
 Player 1 rolls 91 + 92 + 93 and moves to space 10 for a final score, 1000 . 
 
 Since player 1 has at least 1000 points, player 1 wins and the game ends. At this point, the losing player had 745 points and the die had been rolled a total of 993 times; 745 * 993 = 739785 
 . 
 Play a practice game using the deterministic 100-sided die. The moment either player wins, what do you get if you multiply the score of the losing player by the number of times the die was rolled during the game?

(cached)

## Input
```
Player 1 starting pos...
```

### Solve today

`aoc::solve input body`:
    The body is executed as a coroutine. Input is available as the `$input` variable. The results are returned using `[yield]`

In [25]:
set input

Player 1 starting position: 7
Player 2 starting position: 5

In [26]:
proc _detdie {} {
    yield
    set num 0
    while 1 {
        yield [expr {$num % 100 + 1}]
        incr num
        }
}


proc play1 {s1 s2 die} {
    coroutine detdie _detdie
    set rolls 0
    set score(p1) 0
    set score(p2) 0
    set pos(p1) [expr {$s1-1}]
    set pos(p2) [expr {$s2-1}]
    set turn p1
    while 1 {
        set roll 0
        incr roll [$die]
        incr roll [$die]
        incr roll [$die]
        incr rolls 3
        incr pos($turn) $roll
        set pos($turn) [expr {$pos($turn) % 10}]
        incr score($turn) $pos($turn)
        incr score($turn)
        if {$score($turn) >= 1000} {
            if {$turn eq "p1"} {
                set loser p2
            } else {
                set loser p1
            }
            # parray score
            # parray pos
            # puts $rolls
            return [list $score($loser) $rolls]
        }
        if {$turn eq "p1"} {
            set turn p2
        } else {
            set turn p1
        }
    }
}


* {*}[play1 4 8 detdie]

739785

In [27]:
set diracrolls {}
foreach d1 {1 2 3} {
    foreach d2 {1 2 3} {
        foreach d3 {1 2 3} {
            lappend diracrolls [+ $d1 $d2 $d3]
        }
    }
}

In [28]:
array unset p
array unset diracmulti
foreach num $diracrolls {
    incr p($num)
}
set diracmulti [array get p]

8 3 4 3 9 1 5 6 6 7 7 6 3 1

In [29]:

array unset cache

proc play2 {p1 p2} {
    set wins [_play2 [expr {$p1-1}] [expr {$p2-1}] 0 0 p1 1]
    return [lindex [lsort -integer $wins] end]
}


proc _play2 {p1 p2 s_p1 s_p2 turn multi} {
    # puts $state
    set w(p1) 0
    set w(p2) 0
    if {[info exists ::cache($p1,$p2,$s_p1,$s_p2,$turn,$multi)]} {
        # puts $cached
        return $::cache($p1,$p2,$s_p1,$s_p2,$turn,$multi)
    }

        set orgscore [set s_$turn]
        set orgpos [set $turn]
        # puts "$p1 $p2 $s_p1 $s_p2 $turn $orgscore"
        foreach {roll rollmulti} $::diracmulti {
            # parray w
            set newpos [expr {($orgpos + $roll) % 10}]
            set score [+ $orgscore $newpos 1]
            # puts $score
            if {$score >= 21} {
                incr w($turn) $rollmulti
                # puts [string length $::w(p1)]
            } else {
                # puts rec
                # incr depth
                if {$turn eq "p1"} {
                   set result [_play2 $newpos $p2 $score $s_p2 p2 [* $multi $rollmulti]]
                } else {
                   set result [_play2 $p1 $newpos $s_p1 $score p1 [* $multi $rollmulti]]
                }
                # puts $result
                lassign  $result w1 w2
                incr w(p1) [* $w1 $rollmulti]
                incr w(p2) [* $w2 $rollmulti]
            }
        }
        set ::cache($p1,$p2,$s_p1,$s_p2,$turn,$multi) [list $w(p1) $w(p2)]
        return [list $w(p1) $w(p2)]
    
}


In [30]:
aoc::solve $input {
    # $input is available in the body
    # return the results using yield

    yield [* {*}[play1 7 5 detdie]]
    yield [play2 7 5]
} 

Part1	798147 (471 us)
Part2	809953813657517 (10201369 us)
